# --- Data Cleaning & Examination ---

In [1]:
# --- Imports ---
import gc
import sys
from pathlib import Path
import numpy as np
import polars as pl

In [2]:
# --- Resolving Package Path ---
notebook_dir = Path.cwd()
src_path = (notebook_dir / ".." / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --- Importing API Files ---
from random_forest.utils import Encoder, train_test_split

In [3]:
# --- Info ---
help(Encoder)

Help on class Encoder in module random_forest.utils.encoder:

class Encoder(builtins.object)
 |  ​
 |  --- Encoder Class ---
 |  Encoder implemented from scratch by "Sepanta Metanat"
 |
 |  First edit: "2026/08/1"
 |  Last edit: "2026/08/5"
 |
 |  Methods defined here:
 |
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  decoder(...)
 |      ​
 |      #> Usage and Information
 |      Use this function to decode encoded data back to original form.
 |               (based on training info)
 |
 |      #> Parameters Documentation:
 |      1. X: X array
 |      2. columns: Columns to get encoded
 |      3. save_path: Save-path for saving the model info
 |              (to be able to encode new data based on training)
 |
 |  encoder(...)
 |      ​
 |      #> Usage and Information
 |      Use this function to encoding pre-defined columns based on training info.
 |
 |      #> Parameters Documentation:
 |      1. X: X array
 |
 |  fit_transform(
 |

In [4]:
# --- Info ---
help(train_test_split)

Help on function train_test_split in module random_forest.utils.train_test_split:

train_test_split(
    X: Iterable,
    Y: Iterable,
    *,
    test_size: float = 0.2,
    subsample: float | int | None = None,
    shuffle: bool = True,
    random_state: int | None = None
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
    ​
    --- Train/Test Split ---
    Function implemented from scratch by "Sepanta Metanat"

    First edit: "2026/02/25"
    Last edit: "2026/08/5"

    #> Usage and Information
    Use this function to split train/test data while customizing it.

    #> Parameters Documentation:
    1. X: X array
    2. Y: Y array
    3. test_size: Based on total size, the amount dedicated to testing
    4. subsample: Percentage or sample-count to create subsample from total
    5. shuffle: Flag for shuffling the data
    6. random_state: Random state value
        (to be able to encode new data based on training)



## > SAML-D Dataset

In [5]:
# --- Reading Dataset Lazily ---
lf = pl.scan_csv(r'..\data\raw\SAML_D.csv')

In [6]:
# --- Data Cleaning & Type Casting (lazy, streamed at collect time) ---
lf = lf.drop(["Date", "Sender_account", "Receiver_account", "Laundering_type"])
lf = lf.with_columns([
    pl.col("Time").str.strptime(pl.Time, "%H:%M:%S").dt.hour().alias("Time"),
    pl.col("Amount").cast(pl.Float32).round(3),
    pl.col("Is_laundering").cast(pl.Int8)
])
# --- Creating New Columns ---
lf = lf.with_columns(
    Sender_and_Receiver_Mismatch = (pl.col("Sender_bank_location") != pl.col("Receiver_bank_location")).cast(pl.Int8),
    Currency_Mismatch = (pl.col("Payment_currency") != pl.col("Received_currency")).cast(pl.Int8)
)
# --- Writing Processed Data (streamed to disk, bounded memory, single file) ---
lf.sink_parquet(r'..\data\processed\processed_SAML_D.parquet')

In [7]:
# --- Representing Data ---
raw_data_amount = lf.select(pl.len()).collect().item()
print(f"Dataset's Length: {raw_data_amount}")
lf.head(12).collect() #> Showing some part of the dataframe

Dataset's Length: 9504852


Time,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Sender_and_Receiver_Mismatch,Currency_Mismatch
i8,f32,str,str,str,str,str,i8,i8,i8
10,1459.150024,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,0,0
10,6019.640137,"""UK pounds""","""Dirham""","""UK""","""UAE""","""Cross-border""",0,1,1
10,14328.44043,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,0,0
10,11895.0,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,0,0
10,115.25,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,0,0
…,…,…,…,…,…,…,…,…,…
10,56.900002,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Credit card""",0,0,0
10,4738.450195,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,0,0
10,5883.870117,"""Indian rupee""","""UK pounds""","""UK""","""UK""","""Credit card""",0,0,1


In [8]:
# --- Checking Data's Balance (lazy aggregation) ---
lf.group_by("Is_laundering").len().collect()

Is_laundering,len
i8,u32
1,9873
0,9494979


In [9]:
# --- Materializing (bounded via the streaming engine) ---
df = lf.collect(engine="streaming")
Y = df["Is_laundering"].to_numpy().astype(np.bool)
X = df.drop(["Is_laundering"]).to_numpy()
# --- Clean Up ---
del df
gc.collect()

169

In [10]:
# --- Encdoing ---
Encoder = Encoder()
X = Encoder.fit_transform(
    X,
    [2,3,4,5,6],
    save_path = r"..\\artifacts\\encoder\\saml_d\\")

In [11]:
# --- Splitting ---
x_train, x_test, y_train, y_test = train_test_split(
    X.astype(np.float32),
    Y,
    test_size=0.2,
    random_state=42)
# --- Saving Encoding Model ---
Encoder.save_model(
    save_dir_path = r"..\\artifacts\\encoder\\saml_d\\", 
    silent_save = True)
# --- Saving Split ---
np.save(r"..\\data\\splits\\saml_d\\x_train", x_train)
np.save(r"..\\data\\splits\\saml_d\\x_test", x_test)
np.save(r"..\\data\\splits\\saml_d\\y_train", y_train)
np.save(r"..\\data\\splits\\saml_d\\y_test", y_test)

In [12]:
# --- Clean up ---
del Encoder
del X, Y
del x_train, x_test, y_train, y_test
gc.collect()

0

## > SUSY Dataset

In [13]:
column_names = [
    'class_label','lepton_1_pT','lepton_1_eta','lepton_1_phi','lepton_2_pT',
    'lepton_2_eta','lepton_2_phi','missing_energy_magnitude','missing_energy_phi',
    'MET_rel','axial_MET','M_R','M_TR_2','R','MT2','S_R','M_Delta_R','dPhi_r_b','cos']

In [14]:
# --- Reading Dataset Lazily ---
lf = pl.scan_csv(
    r'..\data\raw\SUSY.csv',
    has_header=False,       #> Tells polars the file has no existing column names
    new_columns=column_names)

In [15]:
# --- Editing Y Column (lazy) ---
lf = lf.cast(pl.Float32)
lf = lf.with_columns(pl.col("class_label").cast(pl.Int8))

# --- Writing Processed Data (streamed to disk, bounded memory, single file) ---
lf.sink_parquet(r'..\data\processed\processed_SUSY.parquet')

In [16]:
# --- Representing Data ---
raw_data_amount = lf.select(pl.len()).collect().item()
print(f"Dataset's Length: {raw_data_amount}")
lf.head(12).collect() #> Showing some part of the dataframe

Dataset's Length: 5000000


class_label,lepton_1_pT,lepton_1_eta,lepton_1_phi,lepton_2_pT,lepton_2_eta,lepton_2_phi,missing_energy_magnitude,missing_energy_phi,MET_rel,axial_MET,M_R,M_TR_2,R,MT2,S_R,M_Delta_R,dPhi_r_b,cos
i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,0.972861,0.653855,1.176225,1.157156,-1.739873,-0.874309,0.567765,-0.175,0.810061,-0.252552,1.921887,0.889637,0.410772,1.145621,1.932632,0.994464,1.367815,0.040714
1,1.667973,0.064191,-1.225171,0.506102,-0.338939,1.672543,3.475464,-1.219136,0.012955,3.775174,1.045977,0.568051,0.481928,0.0,0.44841,0.205356,1.321893,0.377584
1,0.44484,-0.134298,-0.709972,0.451719,-1.613871,-0.768661,1.219918,0.504026,1.831248,-0.431385,0.526283,0.941514,1.587535,2.024308,0.603498,1.562374,1.135454,0.18091
1,0.381256,-0.976145,0.693152,0.448959,0.891753,-0.677328,2.03306,1.533041,3.04626,-1.005285,0.569386,1.015211,1.582217,1.551914,0.761215,1.715464,1.492257,0.090719
1,1.309996,-0.690089,-0.676259,1.589283,-0.693326,0.622907,1.087562,-0.381742,0.589204,1.365479,1.179295,0.968218,0.728563,0.0,1.083158,0.043429,1.154854,0.094859
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0.571656,0.205696,0.42595,0.578514,0.927399,-1.101478,0.116555,1.549753,0.174421,0.22345,0.523164,0.289889,0.49171,0.334593,0.514246,0.268207,0.315795,0.125728
1,2.112812,0.742983,-0.330539,0.805253,-0.028887,-1.446679,2.299946,1.450429,2.98911,-1.89477,1.445125,2.548166,1.564721,2.393632,1.554566,2.148468,1.179117,0.688057
0,1.001869,-0.471788,0.555614,1.233368,1.255548,-1.052491,0.437615,-1.333052,0.326858,-0.111678,1.435708,0.755201,0.466779,0.454541,1.446331,0.592259,1.325197,0.083014


In [17]:
# --- Materializing (bounded via the streaming engine) ---
df = lf.collect(engine="streaming")
Y = df["class_label"].to_numpy().astype(np.bool)
X = df.drop("class_label").to_numpy().astype(np.float32)
# --- Clean Up ---
del df
gc.collect()

0

In [18]:
# --- Splitting ---
x_train, x_test, y_train, y_test = train_test_split(X, Y,
                                                    subsample = 0.3,  #> Percentage of the whole dataset
                                                    test_size = 0.2,  #> Test Size
                                                    shuffle = True,
                                                    random_state = 42)

# --- Saving ---
np.save(r"..\\data\\splits\\susy\\x_train", x_train)
np.save(r"..\\data\\splits\\susy\\x_test", x_test)
np.save(r"..\\data\\splits\\susy\\y_train", y_train)
np.save(r"..\\data\\splits\\susy\\y_test", y_test)

In [19]:
# --- Clean up ---
del lf, X, Y
del x_train, x_test, y_train, y_test
gc.collect()

0

## > HIGGS Dataset

In [20]:
# --- Defining Columns ---
column_names = [
    'class_label', 'lepton_pT', 'lepton_eta', 'lepton_phi', 'missing_energy_magnitude', 'missing_energy_phi',
    'jet_1_pt', 'jet_1_eta', 'jet_1_phi', 'jet_1_b-tag', 'jet_2_pt', 'jet_2_eta', 'jet_2_phi', 
    'jet_2_b-tag', 'jet_3_pt', 'jet_3_eta', 'jet_3_phi', 'jet_3_b-tag', 'jet_4_pt', 'jet_4_eta',
    'jet_4_phi', 'jet_4_b-tag', 'm_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb']

In [21]:
# --- Reading Dataset Lazily ---
lf = pl.scan_csv(
    r'..\data\raw\HIGGS.csv',
    has_header=False,       #> Tells polars the file has no existing column names
    new_columns=column_names)

In [22]:
# --- Editing Y Column (lazy) ---
lf = lf.cast(pl.Float32)
lf = lf.with_columns(pl.col("class_label").cast(pl.Int8))

# --- Writing Processed Data (streamed to disk, bounded memory, single file) ---
lf.sink_parquet(r'..\data\processed\processed_HIGGS.parquet')

In [23]:
# --- Representing Data ---
raw_data_amount = lf.select(pl.len()).collect().item()
print(f"Dataset's Length: {raw_data_amount}")
lf.head(12).collect() #> Showing some part of the dataframe

Dataset's Length: 11000000


class_label,lepton_pT,lepton_eta,lepton_phi,missing_energy_magnitude,missing_energy_phi,jet_1_pt,jet_1_eta,jet_1_phi,jet_1_b-tag,jet_2_pt,jet_2_eta,jet_2_phi,jet_2_b-tag,jet_3_pt,jet_3_eta,jet_3_phi,jet_3_b-tag,jet_4_pt,jet_4_eta,jet_4_phi,jet_4_b-tag,m_jj,m_jjj,m_lv,m_jlv,m_bb,m_wbb,m_wwbb
i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1,0.869293,-0.635082,0.22569,0.32747,-0.689993,0.754202,-0.248573,-1.092064,0.0,1.374992,-0.653674,0.930349,1.107436,1.138904,-1.578198,-1.046985,0.0,0.65793,-0.010455,-0.045767,3.101961,1.35376,0.979563,0.978076,0.920005,0.721657,0.988751,0.876678
1,0.907542,0.329147,0.359412,1.49797,-0.31301,1.095531,-0.557525,-1.58823,2.173076,0.812581,-0.213642,1.271015,2.214872,0.499994,-1.261432,0.732156,0.0,0.398701,-1.13893,-0.000819,0.0,0.30222,0.833048,0.9857,0.978098,0.779732,0.992356,0.798343
1,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.0,0.851737,1.540659,-0.81969,2.214872,0.99349,0.35608,-0.208778,2.548224,1.256955,1.128848,0.900461,0.0,0.909753,1.10833,0.985692,0.951331,0.803252,0.865924,0.780118
0,1.344385,-0.876626,0.935913,1.99205,0.882454,1.786066,-1.646778,-0.942383,0.0,2.423265,-0.676016,0.736159,2.214872,1.29872,-1.430738,-0.364658,0.0,0.745313,-0.678379,-1.360356,0.0,0.946652,1.028704,0.998656,0.728281,0.8692,1.026736,0.957904
1,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.0,0.800872,1.020974,0.971407,2.214872,0.596761,-0.350273,0.631194,0.0,0.479999,-0.373566,0.113041,0.0,0.755856,1.361057,0.98661,0.838085,1.133295,0.872245,0.808487
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,0.933895,0.62913,0.527535,0.238033,-0.966569,0.547811,-0.059439,-1.706866,2.173076,0.941003,-2.653733,-0.15722,0.0,1.03037,-0.175505,0.523021,2.548224,1.373547,1.291248,-1.467454,0.0,0.901837,1.083671,0.979696,0.7833,0.849195,0.894356,0.774879
1,1.405144,0.536603,0.689554,1.179567,-0.110061,3.202405,-1.52696,-1.576033,0.0,2.931537,0.567342,-0.130033,2.214872,1.787123,0.899499,0.585151,2.548224,0.401865,-0.151202,1.163489,0.0,1.667071,4.039273,1.175828,1.045352,1.542972,3.534827,2.740754
1,1.176566,0.104161,1.397002,0.479721,0.265513,1.135563,1.534831,-0.253291,0.0,1.027247,0.534316,1.180022,0.0,2.405661,0.087557,-0.976534,2.548224,1.250383,0.268541,0.530334,0.0,0.833175,0.773968,0.98575,1.103696,0.84914,0.937104,0.812364


In [24]:
# --- Materializing (bounded via the streaming engine) ---
df = lf.collect(engine="streaming")
Y = df["class_label"].to_numpy().astype(np.bool)
X = df.drop("class_label").to_numpy().astype(np.float32)
# --- Clean up ---
del df
gc.collect()

0

In [25]:
# --- Splitting ---
x_train, x_test, y_train, y_test = train_test_split(X, Y,
                                                    subsample = 0.2,  #> Percentage of the whole dataset
                                                    test_size = 0.2,  #> Test Size
                                                    shuffle = True,
                                                    random_state = 42)

# --- Saving ---
np.save(r"..\\data\\splits\\higgs\\x_train", x_train)
np.save(r"..\\data\\splits\\higgs\\x_test", x_test)
np.save(r"..\\data\\splits\\higgs\\y_train", y_train)
np.save(r"..\\data\\splits\\higgs\\y_test", y_test)

In [26]:
# --- Clean up ---
del lf, X, Y
del x_train, x_test, y_train, y_test
gc.collect()

0